# Analyse von ECMAScript-Parsing-Problemen

Dieses Notebook lädt alle relevanten TypeScript/TSX-Dateien im Projekt und versucht, sie mit einer ECMAScript-kompatiblen Parser-Bibliothek zu parsen. Fehler werden gesammelt und auf problematische Dateien untersucht.

In [6]:
import os
from pathlib import Path

root = Path('/workspaces/kreditkarten-vergleich')

file_paths = sorted(root.glob('**/*'))
source_files = [p for p in file_paths if p.suffix in {'.ts', '.tsx', '.js', '.jsx'} and 'node_modules' not in p.parts]
print(f'Gefundene Quelltextdateien: {len(source_files)}')
for p in source_files:
    print(p.relative_to(root))

Gefundene Quelltextdateien: 198
.next/build/chunks/[root-of-the-server]__51225daf._.js
.next/build/chunks/[root-of-the-server]__974941ed._.js
.next/build/chunks/[turbopack-node]_transforms_postcss_ts_6920245c._.js
.next/build/chunks/[turbopack]_runtime.js
.next/build/chunks/node_modules_fe693df6._.js
.next/build/postcss.js
.next/server/app/_global-error/page.js
.next/server/app/_global-error/page_client-reference-manifest.js
.next/server/app/_not-found/page.js
.next/server/app/_not-found/page_client-reference-manifest.js
.next/server/app/ausland/page.js
.next/server/app/ausland/page_client-reference-manifest.js
.next/server/app/datenschutz/page.js
.next/server/app/datenschutz/page_client-reference-manifest.js
.next/server/app/favicon.ico/route.js
.next/server/app/impressum/page.js
.next/server/app/impressum/page_client-reference-manifest.js
.next/server/app/kreditkarte-studenten-ausland-kostenlos/page.js
.next/server/app/kreditkarte-studenten-ausland-kostenlos/page_client-reference-man

In [3]:
import sys

try:
    from tree_sitter import Language, Parser
    print('tree_sitter bereits installiert')
except Exception:
    print('tree_sitter nicht installiert, installiere...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tree_sitter'])
    from tree_sitter import Language, Parser
    print('tree_sitter installiert')


tree_sitter nicht installiert, installiere...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 23.1 MB/s  0:00:00
tree_sitter installiert



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import sys
import subprocess

for pkg in ['tree_sitter_javascript', 'tree_sitter_typescript']:
    try:
        __import__(pkg)
        print(f'{pkg} bereits installiert')
    except ImportError:
        print(f'Installiere {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        print(f'{pkg} installiert')


Installiere tree_sitter_javascript...
tree_sitter_javascript installiert
Installiere tree_sitter_typescript...



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


tree_sitter_typescript installiert



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [25]:
from pathlib import Path
from tree_sitter import Parser, Language
from tree_sitter_javascript import language as javascript_language
from tree_sitter_typescript import language_tsx, language_typescript

root = Path('/workspaces/kreditkarten-vergleich')
source_files = sorted([p for p in root.glob('**/*') if p.suffix in {'.ts', '.tsx', '.js', '.jsx'} and 'node_modules' not in p.parts])

js_parser = Parser()
js_parser.language = Language(javascript_language())

ts_parser = Parser()
ts_parser.language = Language(language_typescript())

tsx_parser = Parser()
tsx_parser.language = Language(language_tsx())

errors = []
for path in source_files:
    text = path.read_text(encoding='utf-8')
    parser = js_parser
    if path.suffix == '.ts':
        parser = ts_parser
    elif path.suffix == '.tsx':
        parser = tsx_parser
    try:
        tree = parser.parse(bytes(text, 'utf8'))
        if tree is None or tree.root_node.has_error:
            # Try to get more detailed error info
            try:
                error_details = []
                if tree and tree.root_node.has_error:
                    # Walk the tree to find error nodes
                    def find_errors(node, depth=0):
                        if node.has_error and node.type != 'program':
                            error_details.append(f"Error at line {node.start_point[0]+1}: {node.type}")
                        if depth < 3:  # Limit depth to avoid too much output
                            for child in node.children:
                                find_errors(child, depth + 1)
                    find_errors(tree.root_node)
                errors.append((path.relative_to(root), error_details if error_details else ["Parse tree has errors"]))
            except:
                errors.append((path.relative_to(root), ["Error analyzing parse tree"]))
    except Exception as exc:
        errors.append((path.relative_to(root), [str(exc)]))

print(f'Geprüfte Dateien: {len(source_files)}')
print(f'Fehlerhafte Dateien: {len(errors)}')
if errors:
    print('\nFehlerhafte Dateien:')
    for path, error_list in errors[:5]:  # Show only first 5 to avoid too much output
        print(f'\n{path}:')
        for error in error_list[:3]:  # Show only first 3 errors per file
            print(f'  {error}')

Geprüfte Dateien: 198
Fehlerhafte Dateien: 10

Fehlerhafte Dateien:

app/HomeClient.tsx:
  Error at line 6: export_statement
  Error at line 6: function_declaration
  Error at line 6: statement_block

app/ausland/page.tsx:
  Error at line 9: export_statement
  Error at line 9: function_declaration
  Error at line 9: statement_block

app/kreditkarte-australien-ohne-gebuehren/page.tsx:
  Error at line 85: labeled_statement
  Error at line 85: ERROR

app/kreditkarte-bali-ohne-gebuehren/page.tsx:
  Error at line 85: labeled_statement
  Error at line 85: ERROR

app/kreditkarte-dubai-ohne-gebuehren/page.tsx:
  Error at line 85: labeled_statement
  Error at line 85: ERROR


In [16]:
from pathlib import Path
from tree_sitter import Parser, Language
from tree_sitter_javascript import language as javascript_language
from tree_sitter_typescript import language_tsx, language_typescript

# Debug: Check what the language functions return
print("javascript_language():", type(javascript_language()))
print("language_typescript():", type(language_typescript()))
print("language_tsx():", type(language_tsx()))

# Try creating Language objects
try:
    js_lang = Language(javascript_language())
    print("JS Language created successfully")
except Exception as e:
    print(f"JS Language creation failed: {e}")

try:
    ts_lang = Language(language_typescript())
    print("TS Language created successfully")
except Exception as e:
    print(f"TS Language creation failed: {e}")

try:
    tsx_lang = Language(language_tsx())
    print("TSX Language created successfully")
except Exception as e:
    print(f"TSX Language creation failed: {e}")

javascript_language(): <class 'PyCapsule'>
language_typescript(): <class 'PyCapsule'>
language_tsx(): <class 'PyCapsule'>
JS Language created successfully
TS Language created successfully
TSX Language created successfully


In [9]:
from tree_sitter import Parser
print(dir(Parser))
print(Parser.__doc__)


['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'included_ranges', 'language', 'logger', 'parse', 'print_dot_graphs', 'reset', 'timeout_micros']
A class that is used to produce a :class:`Tree` based on some source code.


In [7]:
import tree_sitter_javascript
import tree_sitter_typescript
print('tree_sitter_javascript:', dir(tree_sitter_javascript)[:50])
print('tree_sitter_typescript:', dir(tree_sitter_typescript)[:50])


tree_sitter_javascript: ['HIGHLIGHTS_QUERY', 'INJECTIONS_QUERY', 'LOCALS_QUERY', 'TAGS_QUERY', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'language']
tree_sitter_typescript: ['__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_binding', '_files', '_get_query', 'language_tsx', 'language_typescript']


In [11]:
import tree_sitter_javascript
import tree_sitter_typescript
print('tree_sitter_javascript.language', tree_sitter_javascript.language)
print('tree_sitter_typescript.language_typescript', tree_sitter_typescript.language_typescript)
print('tree_sitter_typescript.language_tsx', tree_sitter_typescript.language_tsx)
print(type(tree_sitter_javascript.language))


tree_sitter_javascript.language <built-in function language>
tree_sitter_typescript.language_typescript <built-in function language_typescript>
tree_sitter_typescript.language_tsx <built-in function language_tsx>
<class 'builtin_function_or_method'>


In [26]:
import inspect
try:
    print(inspect.signature(tree_sitter_javascript.language))
except ValueError as e:
    print(f"inspect.signature failed for tree_sitter_javascript.language: {e}")
    print("This is a builtin function, signature inspection not possible.")

try:
    print(inspect.signature(tree_sitter_typescript.language_typescript))
except ValueError as e:
    print(f"inspect.signature failed for tree_sitter_typescript.language_typescript: {e}")
    print("This is a builtin function, signature inspection not possible.")

try:
    print(inspect.signature(tree_sitter_typescript.language_tsx))
except ValueError as e:
    print(f"inspect.signature failed for tree_sitter_typescript.language_tsx: {e}")
    print("This is a builtin function, signature inspection not possible.")

inspect.signature failed for tree_sitter_javascript.language: no signature found for builtin <built-in function language>
This is a builtin function, signature inspection not possible.
inspect.signature failed for tree_sitter_typescript.language_typescript: no signature found for builtin <built-in function language_typescript>
This is a builtin function, signature inspection not possible.
inspect.signature failed for tree_sitter_typescript.language_tsx: no signature found for builtin <built-in function language_tsx>
This is a builtin function, signature inspection not possible.
